# A receive-fixed swap: contracts, cashflows, value, and risk

**Question.** Is receiving 4% fixed on a five-year USD swap attractive relative
to this SOFR curve, and can we account for its entire reported value?

We value USD 1 million notional at origination on the illustrative
24 February 2026 market. Both legs pay USD; SOFR supplies projection and USD
collateral discounting. Dates are unadjusted in the contract below, fixed
payments are semiannual, floating payments quarterly, and accrual is Actual/360.
We do not model counterparty default, funding spreads, or future realized rates.
[Setup](../../README.md) · [Detailed par-rate study](par_swap.ipynb).

`Swap` holds contractual terms; `Trade` holds position direction and execution
date. Quotes and curve specifications feed `PricingContext`; requests select
the numerical answers. The helper only loads inputs and never prices a trade.


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market

market = load_market()
ref = market.quotes.reference_date
print(
    "Market date:",
    ref,
)
market.quotes.to_dataframe().head(8)


Market date: 2026-02-24


,identifier,mid,bid,ask
0,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.035_...,0.20,None,None
1,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.045_...,0.18,None,None
2,CapletFloorlet_USD_SOFR_3M_10Y_Absolute_0.055_...,0.20,None,None
3,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.035_...,0.19,None,None
4,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.045_...,0.17,None,None
5,CapletFloorlet_USD_SOFR_3M_12Y_Absolute_0.055_...,0.19,None,None
6,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.035_S...,0.33,None,None
7,CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_S...,0.29,None,None


## Dates and conventions carry financial meaning

`Date` is a calendar date; adding a `Period` gives another date. A date shift
does not itself apply a business-day convention. Use a `Calendar` when that is
the convention you intend. A `DayCounter` turns an accrual interval into a year
fraction. A rate of `0.04` means 4%; `0.0001` is one basis point.

Typed enums make Python contracts readable and discoverable in an editor.
Strings are useful in JSON files, but you do not need to encode your Python
product as a large string-valued dictionary.

In [2]:
six_months = qs.Period(
    6,
    qs.TimeUnit.Months,
)
calendar = qs.Calendar("UnitedStates(Sofr)")
end = ref + six_months
print(
    "Unadjusted end:",
    end,
)
print(
    "Adjusted end:",
    calendar.adjust(
        end,
        qs.BusinessDayConvention.ModifiedFollowing,
    ),
)
print(
    "Actual/360 accrual:",
    qs.DayCounter.Actual360.year_fraction(
        ref,
        end,
    ),
)
print(
    "Receive / pay signs:",
    qs.Side.LongReceive.sign(),
    qs.Side.PayShort.sign(),
)


Unadjusted end: 2026-08-24
Adjusted end: 2026-08-24
Actual/360 accrual: 0.5027777777777778
Receive / pay signs: 1.0 -1.0


## Identify a curve before asking for it

`MarketIndex.SOFR` is a lookup key shared by curve configurations, products,
volatility objects, and rate models. It is not a rate value. The curve below is
built from a deposit and OIS quotes. Those quoted instruments are its *pillars*;
the resulting dated discount factors are its *nodes*.

`SingleCurveCSADiscountPolicy` sets the context's base currency and base index. In this
single-currency example SOFR supplies both projection and discounting. A
multi-curve setup can distinguish those roles.

In [3]:
print(market.csa_policy)
market.curves[0].to_dict()


SingleCurveCSADiscountPolicy(collateral_currency=USD, collateral_index=SOFR)


{'market_index': 'SOFR',
 'construction': {'type': 'Bootstrapped'},
 'day_counter': 'Actual360',
 'interpolator': 'LogLinear',
 'enable_extrapolation': True,
 'quotes': ['FixedRateDeposit_USD_SOFR_1D',
  'OIS_USD_SOFR_1Y',
  'OIS_USD_SOFR_2Y',
  'OIS_USD_SOFR_3Y',
  'OIS_USD_SOFR_5Y',
  'OIS_USD_SOFR_7Y',
  'OIS_USD_SOFR_10Y',
  'OIS_USD_SOFR_30Y']}

## Describe the position

`LongReceive` receives fixed and pays floating. Position direction is on the
explicit `Trade`; instrument terms and trade metadata are separate. The
identifier lets us join results back to the book.


In [4]:
swap = qs.Swap(
    identifier="USD-RECEIVE-FIXED-5Y",
    start_date=ref,
    maturity_date=ref + "5Y",
    notional=1_000_000,
    fixed_rate=0.04,
    currency=qs.Currency.USD,
    market_index=qs.MarketIndex.SOFR,
    day_counter=qs.DayCounter.Actual360,
    fixed_leg_frequency=qs.Frequency.Semiannual,
    floating_leg_frequency=qs.Frequency.Quarterly,
)
trade = qs.Trade(swap, ref, qs.Side.LongReceive)
trade

Trade(instrument=Swap(identifier='USD-RECEIVE-FIXED-5Y', start_date='2026-02-24', maturity_date='2031-02-24', notional=1000000, fixed_rate=0.04), trade_date='2026-02-24', side=LongReceive)

## Build once, ask for several answers

First construct `context`, then enter `with context:` to activate request-driven market construction and start the automatic-differentiation
tape used for sensitivities. `evaluate` chooses the native swap pricer.
Ordinary swaps need curves, so we do not attach the volatility inputs here.

Use constructed views such as `context.curve(...)` inside this block. Save
numeric values and pandas tables to inspect later: on exit the context clears
its constructed market and rewinds the tape. Avoid nesting active contexts;
scenario examples below use separate, sequential contexts.

In [5]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
)

with context:
    sofr = context.curve(qs.MarketIndex.SOFR)
    five_year_df = sofr.discount_factor(ref + "5Y")
    nodes = sofr.nodes()
    result = context.evaluate(
        trade,
        [
            qs.Request.Value,
            qs.Request.FairRate,
            qs.Request.Cashflows,
            qs.Request.Sensitivities,
        ],
    )
    cashflows = result.cashflows
    risk = result.sensitivities

pd.Series(
    {
        "position_pv_usd": result.price,
        "par_fixed_rate": result.fair_rate,
        "five_year_discount_factor": five_year_df,
    }
)

position_pv_usd             -1664.892276
par_fixed_rate                  0.040368
five_year_discount_factor       0.817367
dtype: float64

## Read the result in its own units

`price` is the signed **position PV**, not a rate or a price per 100. `fair_rate`
is the fixed coupon that makes this swap worth zero on this market. The cashflow
table describes payments; floating amounts are projected using the curve and
are not promises of the rates that will be realized. They are undiscounted
amounts, so summing them does not give the PV.

Results only contain the outputs their pricer supplies. For example, this swap
has no bond yield-to-maturity. Request support is product-specific; see
[valuation catalog](../README.md).

In [6]:
cashflows[["payment_date", "leg_index", "type", "amount", "currency"]].head(10)


,payment_date,leg_index,type,amount,currency
0,2026-02-24,0,Disbursement,1000000.000000,USD
1,2031-02-24,0,Redemption,1000000.000000,USD
2,2026-08-24,0,FixedRateCoupon,20111.111111,USD
3,2027-02-24,0,FixedRateCoupon,20444.444444,USD
4,2027-08-24,0,FixedRateCoupon,20111.111111,USD
5,2028-02-24,0,FixedRateCoupon,20444.444444,USD
6,2028-08-24,0,FixedRateCoupon,20222.222222,USD
7,2029-02-24,0,FixedRateCoupon,20444.444444,USD
8,2029-08-24,0,FixedRateCoupon,20111.111111,USD
9,2030-02-24,0,FixedRateCoupon,20444.444444,USD


The sensitivity table maps the PV back to the original quote identifiers.
`value` is a derivative per **one full unit of the input quote**, not already a
basis-point risk. All inputs below are decimal interest rates, so multiplying
by `1e-4` estimates the USD P&L for a +1 bp move in each quote separately.
The sign is retained: this is a signed quote DV01, not an absolute duration.

In [7]:
quote_risk = risk.assign(pnl_for_plus_1bp_usd=risk["value"] * 1e-4)
quote_risk.sort_values("pnl_for_plus_1bp_usd")


,pillar,value,pnl_for_plus_1bp_usd
6,OIS_USD_SOFR_5Y,-4.418329e+06,-441.832926
5,OIS_USD_SOFR_3Y,-6.315454e+03,-0.631545
2,OIS_USD_SOFR_1Y,-3.049831e+03,-0.304983
0,FixedRateDeposit_USD_SOFR_1D,-2.772806e+03,-0.277281
1,OIS_USD_SOFR_10Y,0.000000e+00,0.000000
4,OIS_USD_SOFR_30Y,0.000000e+00,0.000000
7,OIS_USD_SOFR_7Y,0.000000e+00,0.000000
3,OIS_USD_SOFR_2Y,1.127620e+03,0.112762


## Account for every dollar of PV

For leg signs $s_\ell$, $V=\sum_{\ell,i}s_\ell C_{\ell i}P(0,T_{\ell i})$.
The native swap cashflow table contains **unsigned contractual leg amounts**:
leg 0 is fixed and leg 1 floating. Apply the trade's direction once, then
discount. Principal exchanges at origination and maturity cancel across legs.
This is not the same as summing unsigned payments in a portfolio report.


In [8]:
import math

reconciliation = cashflows.copy()
reconciliation["leg_sign"] = reconciliation["leg_index"].map(
    {0: trade.side.sign(), 1: -trade.side.sign()}
)
assert reconciliation["leg_sign"].notna().all()
reconciliation["signed_amount_usd"] = reconciliation["amount"] * reconciliation["leg_sign"]
with context:
    curve = context.curve(qs.MarketIndex.SOFR)
    reconciliation["discount_factor"] = reconciliation["payment_date"].map(curve.discount_factor)
    opposite = context.evaluate(qs.Trade(swap, ref, qs.Side.PayShort), [qs.Request.Value])
reconciliation["pv_usd"] = reconciliation["signed_amount_usd"] * reconciliation["discount_factor"]
assert math.isclose(reconciliation["pv_usd"].sum(), result.price, abs_tol=1e-6)
assert math.isclose(opposite.price, -result.price, abs_tol=1e-6)
print(f"Receive-fixed PV: USD {result.price:,.2f}; par coupon: {result.fair_rate:.6%}.")
print(f"Cashflow reconstruction residual: USD {reconciliation['pv_usd'].sum() - result.price:.3e}.")
print("The 4% contract is above par." if result.fair_rate < 0.04 else "The 4% contract is below par.")
reconciliation.groupby("leg_index")[["signed_amount_usd", "pv_usd"]].sum()


Receive-fixed PV: USD -1,664.89; par coupon: 4.036800%.
Cashflow reconstruction residual: USD -6.003e-11.
The 4% contract is below par.


,signed_amount_usd,pv_usd
leg_index,,
0,2.202889e+06,1.998335e+06
1,-2.202698e+06,-2.000000e+06


## Conclusion

The signed cashflow reconstruction and reversed-position check explain the
native value independently of the reporting API. The difference between the
agreed and par coupon drives the receive-fixed position's value; future
floating cashflows remain projections, not guaranteed income. Quote DV01
describes small input moves, not realized P&L under arbitrary stresses.

Continue with [par-rate mathematics](par_swap.ipynb),
[independent market shocks](../../volatilitysurface/curves_and_volatility.ipynb),
or [portfolio risk](../../sensitivity/portfolio_risk.ipynb).
